# ⚡ Notebook 3: BiGRU + FastText — Impact Level Prediction

**Architecture:** Bidirectional GRU with Bahdanau Attention + Numerical Feature Fusion

**Task:** Impact Level Prediction — High / Medium / Low

**Key techniques:**
- FastText-style embeddings (trained from scratch on vocab)
- Bahdanau attention to focus on important headline tokens
- Feature fusion: text features + numerical features (Index_Change_Percent, Trading_Volume)
- fp16 mixed precision + early stopping

In [ ]:
!pip install torch scikit-learn matplotlib seaborn pandas numpy

In [ ]:
# ── Import libraries ─────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 📂 Step 1: Load Data & Encode Labels

In [ ]:
# ── Load cleaned dataset ─────────────────────────────────────────────────────
df = pd.read_csv("data/processed/financial_news_clean.csv")
print(f"Dataset loaded: {df.shape}")

# ── Encode Impact Level: High=0, Low=1, Medium=2 (alphabetical by default) ───
le_impact = LabelEncoder()
df["impact_label"] = le_impact.fit_transform(df["Impact_Level"])
NUM_CLASSES = len(le_impact.classes_)
print(f"Impact Level classes: {le_impact.classes_}")

# ── Stratified split to ensure all impact levels in each split ───────────────
train_df, temp_df = train_test_split(df, test_size=0.30,
                                     stratify=df["impact_label"], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50,
                                     stratify=temp_df["impact_label"], random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 🔤 Step 2: Build Vocabulary & Tokenizer

In [ ]:
# ── Simple whitespace tokenizer ──────────────────────────────────────────────
# Lowercase, remove punctuation, split by space
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)  # keep only alphanumeric + space
    return text.split()

# ── Build vocabulary from TRAINING set only ──────────────────────────────────
# Important: never use val/test data to build vocab (data leakage!)
all_tokens = [token for headline in train_df["Headline"]
              for token in tokenize(str(headline))]

# Keep top 30,000 most frequent tokens
counter = Counter(all_tokens)
vocab = {word: idx + 2 for idx, (word, _) in enumerate(counter.most_common(30000))}
vocab["<PAD>"] = 0   # padding token: used to fill short sequences
vocab["<UNK>"] = 1   # unknown token: for words not in vocab

VOCAB_SIZE = len(vocab)
MAX_LEN = 64  # max tokens per headline (covers 95%+ of headlines)
print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Max sequence length: {MAX_LEN}")

def encode_headline(text, max_len=MAX_LEN):
    """Convert headline string to padded list of token IDs."""
    tokens = tokenize(str(text))[:max_len]              # truncate if too long
    ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]  # map to IDs
    ids += [vocab["<PAD>"]] * (max_len - len(ids))      # pad to max_len
    return ids

## 🏗️ Step 3: PyTorch Dataset

In [ ]:
class ImpactLevelDataset(Dataset):
    """
    Dataset for BiGRU Impact Level prediction.
    Returns:
        - X    : encoded headline token IDs (max_len,)
        - num  : normalized numerical features (Index_Change_Percent, Trading_Volume)
        - y    : impact level label (0=High, 1=Low, 2=Medium)
    """
    def __init__(self, df):
        # Encode all headlines to token ID sequences
        self.X = torch.tensor(
            [encode_headline(h) for h in df["Headline"]], dtype=torch.long
        )

        # Stack numerical features: shape (N, 2)
        num_feats = df[["Index_Change_Percent", "Trading_Volume"]].values.astype(float)
        num_tensor = torch.tensor(num_feats, dtype=torch.float32)

        # Z-score normalization: (x - mean) / std
        # Prevents large-scale features from dominating the model
        self.mean = num_tensor.mean(dim=0)
        self.std  = num_tensor.std(dim=0) + 1e-8   # add epsilon to avoid div/0
        self.num  = (num_tensor - self.mean) / self.std

        self.y = torch.tensor(df["impact_label"].values, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.num[idx], self.y[idx]

## 🧠 Step 4: BiGRU Model with Bahdanau Attention

In [ ]:
class BahdanauAttention(nn.Module):
    """
    Bahdanau (Additive) Attention Mechanism.

    Instead of using just the final hidden state, attention allows the model
    to look at ALL hidden states and learn which tokens matter most for
    predicting impact level.

    e.g., for "Fed raises interest rates by 75bps — market crash"
    attention should focus on "raises", "75bps", "crash"
    """
    def __init__(self, hidden_size):
        super().__init__()
        # Learn a score for each time step (each token position)
        self.attn = nn.Linear(hidden_size * 2, 1)

    def forward(self, gru_output):
        """
        Args:
            gru_output: (batch, seq_len, hidden*2) — all BiGRU hidden states
        Returns:
            context    : (batch, hidden*2) — weighted sum of hidden states
            attn_weights: (batch, seq_len) — attention weights (sum to 1)
        """
        # Compute attention score for each token position
        scores = self.attn(gru_output)                       # (B, T, 1)
        attn_weights = torch.softmax(scores, dim=1)          # normalize to sum=1
        context = (attn_weights * gru_output).sum(dim=1)     # weighted sum: (B, H*2)
        return context, attn_weights.squeeze(-1)


class BiGRUImpact(nn.Module):
    """
    Bidirectional GRU for Impact Level prediction.

    Architecture:
        Headline tokens
            |
        Embedding layer (vocab_size -> embed_dim)
            |
        BiGRU (processes left-to-right AND right-to-left)
            |
        Bahdanau Attention (focus on important tokens)
            |
        Feature Fusion (concat attention output + numerical features)
            |
        Classifier (MLP -> 3 classes: High/Medium/Low)
    """
    def __init__(self, vocab_size, embed_dim=300, hidden_size=128,
                 num_classes=3, num_numerical=2, dropout=0.3):
        super().__init__()

        # Embedding: maps token IDs to dense vectors (300-dim like FastText)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # BiGRU: bidirectional captures both past and future context
        # Output dim = hidden_size * 2 (forward + backward concatenated)
        self.gru = nn.GRU(
            embed_dim, hidden_size,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
            num_layers=2   # 2 stacked GRU layers for deeper representations
        )

        # Bahdanau attention over all GRU hidden states
        self.attention = BahdanauAttention(hidden_size)

        # Final classifier: fuses text context + numerical features
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2 + num_numerical, 64),  # concat text+num
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, numerical_features):
        """
        Args:
            x                 : (batch, seq_len) token IDs
            numerical_features: (batch, 2)       Index_Change_Percent, Trading_Volume
        Returns:
            logits      : (batch, num_classes)
            attn_weights: (batch, seq_len)
        """
        # Step 1: Token IDs -> dense embeddings
        embeddings = self.embedding(x)                   # (B, T, embed_dim)

        # Step 2: BiGRU over embeddings
        gru_out, _ = self.gru(embeddings)                # (B, T, hidden*2)

        # Step 3: Bahdanau attention — get weighted context vector
        context, attn_weights = self.attention(gru_out)  # (B, hidden*2)

        # Step 4: Fuse text context with numerical features
        fused = torch.cat([context, numerical_features], dim=1)  # (B, hidden*2+2)

        # Step 5: Classify into impact levels
        logits = self.classifier(fused)                  # (B, num_classes)

        return logits, attn_weights

## 🚀 Step 5: Training

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
BATCH_SIZE = 32
EPOCHS     = 20
PATIENCE   = 3

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_set = ImpactLevelDataset(train_df)
val_set   = ImpactLevelDataset(val_df)
test_set  = ImpactLevelDataset(test_df)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

# ── Model, optimizer, scheduler, loss ────────────────────────────────────────
model     = BiGRUImpact(VOCAB_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ReduceLROnPlateau: halve LR if val F1 does not improve for 2 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5, verbose=True
)
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler()  # fp16

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

In [ ]:
def train_one_epoch(model, loader):
    """One training epoch. Returns average batch loss."""
    model.train()
    total_loss = 0.0
    for X, num, y in loader:
        X, num, y = X.to(DEVICE), num.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():  # fp16
            logits, _ = model(X, num)    # _ = attention weights (not needed in train)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    """Evaluate on val/test set. Returns macro F1, predictions, true labels."""
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for X, num, y in loader:
            X, num = X.to(DEVICE), num.to(DEVICE)
            logits, _ = model(X, num)
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_true.extend(y.numpy())
    macro_f1 = f1_score(all_true, all_preds, average="macro")
    return macro_f1, all_preds, all_true


# ── Training loop with early stopping ────────────────────────────────────────
best_val_f1      = 0.0
patience_counter = 0
train_losses     = []
val_f1_history   = []

print(f"{'Epoch':<8} {'Loss':<12} {'Val F1'}")
print("-" * 30)

for epoch in range(EPOCHS):
    loss     = train_one_epoch(model, train_loader)
    val_f1, _, _ = evaluate(model, val_loader)

    train_losses.append(loss)
    val_f1_history.append(val_f1)
    scheduler.step(val_f1)  # adjust LR based on val F1

    print(f"{epoch+1:<8} {loss:<12.4f} {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "models/saved/bigru_best.pt")
        patience_counter = 0
        print(f"         -> Best model saved! (F1={best_val_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"
Early stopping at epoch {epoch+1}.")
            break

print(f"
Best Validation F1: {best_val_f1:.4f}")

## 📊 Step 6: Evaluation & Attention Visualization

In [ ]:
# ── Load best model and evaluate on test set ─────────────────────────────────
model.load_state_dict(torch.load("models/saved/bigru_best.pt"))
test_f1, preds, true = evaluate(model, test_loader)

print("=" * 55)
print("TASK: Impact Level Prediction — Test Set Results")
print("=" * 55)
print(classification_report(true, preds, target_names=le_impact.classes_))
print(f"Cohen Kappa Score  : {cohen_kappa_score(true, preds):.4f}")
print(f"Matthews Corr Coef : {matthews_corrcoef(true, preds):.4f}")
print(f"Macro F1 Score     : {test_f1:.4f}")

# ── Confusion matrix ──────────────────────────────────────────────────────────
plt.figure(figsize=(7, 5))
sns.heatmap(
    confusion_matrix(true, preds),
    annot=True, fmt="d", cmap="Blues",
    xticklabels=le_impact.classes_,
    yticklabels=le_impact.classes_
)
plt.title("BiGRU — Impact Level Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("results/plots/bigru_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# ── Attention Weight Visualization ───────────────────────────────────────────
# Shows which words in the headline the model focuses on most
# Helps explain model predictions (interpretability)

model.eval()

# Pick a sample headline from test set
sample_idx = 0
sample_headline = test_df["Headline"].iloc[sample_idx]
actual_label    = le_impact.classes_[test_df["impact_label"].iloc[sample_idx]]

tokens = tokenize(str(sample_headline))[:MAX_LEN]
X_sample  = torch.tensor([encode_headline(sample_headline)], dtype=torch.long).to(DEVICE)
num_sample = torch.zeros(1, 2).to(DEVICE)   # use zero numerical features for demo

with torch.no_grad():
    logits, attn_weights = model(X_sample, num_sample)
    predicted_label = le_impact.classes_[logits.argmax(dim=1).item()]

# Extract attention weights for actual tokens (not padding)
attn = attn_weights[0, :len(tokens)].cpu().numpy()

# Plot
plt.figure(figsize=(14, 4))
bars = plt.bar(range(len(tokens)), attn, color="#3498db", alpha=0.8)
plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right", fontsize=10)
plt.title(
    f"Bahdanau Attention Weights
"
    f"Headline: {sample_headline[:80]}...
"
    f"Actual: {actual_label} | Predicted: {predicted_label}",
    fontsize=10
)
plt.ylabel("Attention Weight")
plt.tight_layout()
plt.savefig("results/plots/bigru_attention_weights.png", dpi=150)
plt.show()
print(f"Top attended tokens: {sorted(zip(attn, tokens), reverse=True)[:5]}")

In [ ]:
# ── Save metrics ─────────────────────────────────────────────────────────────
metrics = {
    "Model"           : "BiGRU",
    "Impact_Macro_F1"  : round(f1_score(true, preds, average="macro"), 4),
    "Impact_Weighted_F1": round(f1_score(true, preds, average="weighted"), 4),
    "Impact_Kappa"     : round(cohen_kappa_score(true, preds), 4),
    "Impact_MCC"       : round(matthews_corrcoef(true, preds), 4),
}
pd.DataFrame([metrics]).to_csv("results/metrics/bigru_metrics.csv", index=False)
print("Metrics saved to results/metrics/bigru_metrics.csv")
print(pd.DataFrame([metrics]).T)